### Setup
Autoreload and imports

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Imports
import sys
from pathlib import Path
from art.attacks.evasion import FastGradientMethod, ProjectedGradientDescent, SaliencyMapMethod, CarliniL2Method

sys.path.append(str(Path.cwd().parents[2]))

from utils.functions import get_windowed_data
from utils.notebook import get_model_classifier, clean_data_test, adv_test, FilenameLoader

/opt/anaconda3/envs/reu/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Define Inputs

In [3]:
## Inputs
checkpoint_name, data_name, save_name = FilenameLoader.rand_pos()

checkpoint_file= f"../../../saved_models/{checkpoint_name}"
data_file = f"../../../data/{data_name}"
save_path = f"../final_data/{save_name}"

collapsed = True # True for JSMA and CW
save_clean = False

### Load Model and Data
Load the data and model and run on clean data to check the accuracy/F1.

In [4]:
## Load model and data
# Load original model and ART-wrapper classifier
model, classifier = get_model_classifier(checkpoint_file, collapsed)

# Load data
(x_train, y_train), (x_test, y_test), fed_dataset, scaler = get_windowed_data(data_file, 
                                                                      normalize=True, 
                                                                      train_perc=80)

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
/opt/anaconda3/envs/reu/lib/python3.11/site-packages/pytorch_lightning/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Checkpoint path exists!


In [5]:
## Run clean data test
out = clean_data_test(model = model, classifier = classifier, # model information
                x_test=x_test, y_test=y_test, # data information
                checkpoint_file=checkpoint_file, data_file=data_file, # to save in json
                save_path=save_path, filename="clean.json", # saving information
                save_results=save_clean,
                collapsed=collapsed
                ) 

# Check if output passes
print('noWrapper:', 'PASS' if out['noWrapper']['f1'] > 0.8 else 'FAIL')
print('wrapper:', 'PASS' if out['wrapper']['f1'] > 0.8 else 'FAIL')

save=False, Metrics not saved
Metrics {'noWrapper': {'accuracy': 0.9863296840687964, 'recall': 0.9547720528729431, 'precision': 0.9991784882489942, 'f1': 0.976470669379591, 'falseNegativeRate': 0.04522794712705692, 'falsePositiveRate': 0.0003317978655477515, 'TP': 353934, 'TN': 876749, 'FP': 291, 'FN': 16766}, 'wrapper': {'accuracy': 0.9936525237629634, 'precision': 1.0, 'recall': 0.9786350148367953, 'f1': 0.9892021595680864, 'falseNegativeRate': 0.021364985163204748, 'falsePositiveRate': 0.0, 'TP': 36278, 'TN': 87704, 'FP': 0, 'FN': 792}, 'files': {'checkpointFile': '../../../saved_models/RandomPos-final.ckpt', 'dataFile': '../../../data/RandomPos_0709.csv'}}
noWrapper: PASS
wrapper: PASS


### Adversarial Test
Run adversarial attacks and save the outputted metrics

In [6]:
# running: randpos

for i in range(1, 11):
    gamma = float(i/10)
    adv_test(
        classifier = classifier, # classifier
        x_test=x_test, y_test=y_test, # test data
        checkpoint_file=checkpoint_file, data_file=data_file, # for json
        
        end_index = len(y_test.numpy()),
        path = f"{save_path}/jsma",
        filename=f"adv_gamma_{gamma}.json",
        collapsed=collapsed,
        Attack=SaliencyMapMethod,
        gamma=gamma, # kwargs
        )

=== Attack: SaliencyMapMethod, kwargs: {'gamma': 0.1} ===


python(87330) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
JSMA: 100%|██████████| 124774/124774 [00:36<00:00, 3380.50it/s]


Accuracy:           0.9963
Precision:          1.0000
Recall:             0.9875
F1:                 0.9937
ASR (FNR):          0.0125
False Positive Rate:0.0000
TP=36606, TN=87704, FP=0, FN=464
Time elapsed:       188.56s
Saved metrics to ../final_data/randpos/jsma/adv_gamma_0.1.json
=== Attack: SaliencyMapMethod, kwargs: {'gamma': 0.2} ===


JSMA: 100%|██████████| 124774/124774 [01:43<00:00, 1209.05it/s]


Accuracy:           0.9973
Precision:          1.0000
Recall:             0.9909
F1:                 0.9954
ASR (FNR):          0.0091
False Positive Rate:0.0000
TP=36732, TN=87704, FP=0, FN=338
Time elapsed:       240.07s
Saved metrics to ../final_data/randpos/jsma/adv_gamma_0.2.json
=== Attack: SaliencyMapMethod, kwargs: {'gamma': 0.3} ===


...

Accuracy:           1.0000
Precision:          1.0000
Recall:             1.0000
F1:                 1.0000
ASR (FNR):          0.0000
False Positive Rate:0.0000
TP=37070, TN=87704, FP=0, FN=0
Time elapsed:       443.14s
Saved metrics to ../final_data/randpos/jsma/adv_gamma_0.8.json
=== Attack: SaliencyMapMethod, kwargs: {'gamma': 0.9} ===


JSMA: 100%|██████████| 124774/124774 [02:46<00:00, 749.87it/s] 


Accuracy:           1.0000
Precision:          1.0000
Recall:             1.0000
F1:                 1.0000
ASR (FNR):          0.0000
False Positive Rate:0.0000
TP=37070, TN=87704, FP=0, FN=0
Time elapsed:       282.87s
Saved metrics to ../final_data/randpos/jsma/adv_gamma_0.9.json
=== Attack: SaliencyMapMethod, kwargs: {'gamma': 1.0} ===


JSMA: 100%|██████████| 124774/124774 [02:50<00:00, 731.39it/s] 


Accuracy:           1.0000
Precision:          1.0000
Recall:             1.0000
F1:                 1.0000
ASR (FNR):          0.0000
False Positive Rate:0.0000
TP=37070, TN=87704, FP=0, FN=0
Time elapsed:       289.45s
Saved metrics to ../final_data/randpos/jsma/adv_gamma_1.0.json
